### **Scenario:**
You're a data engineer intern at a growing e-commerce startup. The analytics team handed you a CSV file with **8 million transaction records** from the past year.  

Unfortunately... it’s a mess:
- The data is inconsistent
- Column types are incorrect
- There are missing values, duplicates, mixed date formats
- And it's eating **way too much memory**

Your mission is to clean it up while keeping everything **memory-efficient and performant**.

---

### **Dataset Preview:**
The file is called `transactions.csv` (4 million rows) and contains:

| Column             | Description                                         |
|--------------------|-----------------------------------------------------|
| `customer_id`      | ID of the customer (some are missing)              |
| `transaction_id`   | Unique transaction string like `TXN1234567`        |
| `purchase_amount`  | Sometimes a float, sometimes a string, sometimes blank |
| `currency`         | Should be all 'USD', but has lowercase/missing     |
| `purchase_date`    | Mixed formats like `'2023/01/01'`, `'01-02-2023'`  |
| `product_id`       | Product ID, might be null                          |
| `product_category` | Category like 'Electronics', 'Books', messy casing |
| `is_returned`      | Values like `'yes'`, `True`, `'no'`, `False`, NaN  |

---

In [1]:
import pandas as pd
import psutil
import os

<hr>

**STEP 1:** To understand the memory impact, I’ll load the full dataset of 8 million rows and observe how much memory is used on a machine with 12GB of RAM. 

In [2]:
# Memory usage before reading the file
process = psutil.Process(os.getpid())
print(f"The memory usage before reading the file is: {process.memory_info().rss / (1024**2)} MB")

# read the file
df = pd.read_csv('data/raw/transactions.csv')

# Memory usage after reading the file
process = psutil.Process(os.getpid())
print(f"The memory usage after reading the file is: {process.memory_info().rss / (1024**2)} MB")

The memory usage before reading the file is: 117.203125 MB
The memory usage after reading the file is: 1097.21875 MB


In [3]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000000 entries, 0 to 7999999
Data columns (total 8 columns):
 #   Column            Dtype  
---  ------            -----  
 0   customer_id       float64
 1   transaction_id    object 
 2   purchase_amount   float64
 3   currency          object 
 4   purchase_date     object 
 5   product_id        object 
 6   product_category  object 
 7   is_returned       object 
dtypes: float64(2), object(6)
memory usage: 2.4 GB


**Memory Impact of Loading Full Dataset**

The memory usage before loading the file was **117.2 MB**, and after loading the full 8 million rows, it climbed up to **2.4 GB** — consuming over **1 GB of RAM**.

This shows that reading the entire dataset into memory on a machine with just 4GB of RAM would likely result in severe performance issues — including system freezing, crashing, or shutting down entirely. Obviously, this is not ideal for production environments or personal machines with limited memory resources.

This highlights the need for more **memory-efficient** data loading techniques, such as reading in chunks.




<hr>

**STEP 2: Restart the Kernel & Load a 100,000-Row Sample**

In this step, the kernel is restarted to ensure a fresh memory state. Then, a sample of the first 100,000 rows from the dataset is loaded.

Memory usage is recorded before and after loading the sample to assess the impact.

This subset will be used to explore, clean, and transform the data — allowing us to understand data quality, formats, and structure before scaling the logic to the full dataset.

In [3]:
# Memory usage before first 100k rows
process = psutil.Process(os.getpid())
print(f"The memory usage before reading the file is: {process.memory_info().rss / (1024**2)} MB")

# read the file
df = pd.read_csv('data/raw/transactions.csv', nrows=100000)

# Memory usage after first 100k rows
process = psutil.Process(os.getpid())
print(f"The memory usage of the first 100k rows is: {process.memory_info().rss / (1024**2)} MB")



The memory usage before reading the file is: 117.42578125 MB
The memory usage of the first 100k rows is: 130.9765625 MB


In [4]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 8 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   customer_id       100000 non-null  int64  
 1   transaction_id    100000 non-null  object 
 2   purchase_amount   71209 non-null   float64
 3   currency          66403 non-null   object 
 4   purchase_date     80096 non-null   object 
 5   product_id        80176 non-null   object 
 6   product_category  100000 non-null  object 
 7   is_returned       80037 non-null   object 
dtypes: float64(1), int64(1), object(6)
memory usage: 31.2 MB


**Memory Usage Observation**

Before reading the file, memory usage was approximately **117.63 MB**, and after loading *100,000 rows*, it increased to **130.37 MB**. Interestingly, running the same operation on a *4GB RAM machine* previously consumed over **300 MB**, whereas on a *12GB RAM machine*, the memory footprint was more than halved.

This highlights how hardware resources can significantly impact memory efficiency, even when reading data in chunks. It also reinforces that chunked reading is both necessary and more effective on machines with higher memory capacity.


<hr>

**STEP 3: Explore, Understand & Transform the Data**

In this step, we will explore the dataset to understand its structure, identify inconsistencies, and assess data quality. Based on these insights, we’ll clean and transform the data — with a strong focus on optimizing memory usage through proper data type conversions and handling of missing or inconsistent values.

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [5]:
# Drop duplicates
df = df.drop_duplicates()

In [6]:
# View first 10 rows
df.head(10)

,customer_id,transaction_id,purchase_amount,currency,purchase_date,product_id,product_category,is_returned
0,25794,TXN2867825,120.50,USD,2023-03-15,P_234,books,no
1,10859,TXN1419610,NaN,usd,2023/01/01,P_234,Books,no
2,86819,TXN5614226,-15.00,USD,2023/01/01,P_456,toys,True
3,64885,TXN5108603,-15.00,USD,03.04.2023,P_234,Books,True
4,16264,TXN4744854,45.99,USD,03.04.2023,P_456,Electronics,True
5,92385,TXN3341057,120.50,USD,NaN,P_234,TOYS,True
6,47193,TXN2719583,NaN,usd,01-02-2023,P_456,Electronics,yes
7,97497,TXN2458591,45.99,USD,2023/01/01,NaN,books,False
8,54130,TXN8078673,NaN,NaN,2023-03-15,P_234,Books,False
9,70262,TXN1533224,60.00,USD,NaN,P_456,TOYS,yes


From the first 10 rows of the 100,000-row chunk, a significant number of NaN values appear in the *purchase_amount* column. The column is of *float64 dtype*, which suggests that missing or non-numeric values were likely parsed as NaN. While it's unlikely that any actual strings were misinterpreted as NaN, we’ll double-check to confirm there are no unexpected values.

To investigate further, I’ll reload the CSV but this time with only the 100k rows. Since a few NaNs appear within that range, it will serve as a suitable sample for closer inspection.

In [7]:
df2 = pd.read_csv('data/raw/transactions.csv', nrows=100000, dtype={"purchase_amount": "object"})


In [8]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 8 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   customer_id       100000 non-null  int64 
 1   transaction_id    100000 non-null  object
 2   purchase_amount   71209 non-null   object
 3   currency          66403 non-null   object
 4   purchase_date     80096 non-null   object
 5   product_id        80176 non-null   object
 6   product_category  100000 non-null  object
 7   is_returned       80037 non-null   object
dtypes: int64(1), object(7)
memory usage: 6.1+ MB


In [8]:
print("sum of nulls in df1 purchase amount column",df['purchase_amount'].isna().sum())
print("sum of nulls in df2 purchase amount column",df2['purchase_amount'].isna().sum())


sum of nulls in df1 purchase amount column 28791
sum of nulls in df2 purchase amount column 28791


The number of null values in the purchase_amount column remains consistent across different data types. Both versions of the DataFrame—one where the column was parsed as float and another where it was cast as object—report exactly 28,791 missing values. 

This confirms that the NaN entries are genuine and not the result of hidden string values or parsing inconsistencies. The total count of nulls remains unchanged across both data types.

**Transform *purchase_amount* Column**

The *purchase_amount* column was correctly parsed as a float, so no further transformation is required. The only necessary step is to drop rows containing null values in this column.

In [9]:
# Drop all nulls in purchase_amount column
df.dropna(subset=['purchase_amount'], inplace=True)

In [13]:
# Let's see the number of unique values in the customer id column
df['customer_id'].nunique()

49141

In [11]:
# Convert customer_id to category type
df['customer_id'] = df['customer_id'].astype('Int32').astype('category')


**Transform *transaction_id* Column**

From challenge description, we understand that this column contains distinct values. Therefore, we will convert its data type to a more memory-efficient format without altering its content.



In [12]:
df['transaction_id'] = df['transaction_id'].astype('string')

**Transform *currency* Column**

In this step, we will transform and standardize the *currency* column to ensure consistency across all values.

In [13]:
# check the unique values in the currency column
df['currency'].unique()

array(['USD', nan, 'usd'], dtype=object)

In [14]:
# convert all currency values to uppercase, except for NaN values
df['currency'] = df['currency'].str.replace(r'^usd$', 'USD', case=False, regex=True)


In [37]:
# convert currency to category datatype
df['currency'] = df['currency'].astype('category')

**Transform *purchase_amount* Column**

In this step, we will clean and standardize the purchase_date column. The column contains inconsistent date formats such as DD/MM/YYYY, YYYY/MM/DD, YYYY.MM.DD, and others. We will correct these inconsistencies step by step to ensure uniform formatting and proper date parsing.

In [15]:
# Clean purchase_amount column before type conversion
df['purchase_date'] = df['purchase_date'].str.strip().str.replace('.','/').str.replace('-', '/')

In [16]:
# create a function to parse the date column
from datetime import datetime

def parse_date(value):
    for fmt in ("%Y/%m/%d", "%/m/%d/%Y", "%d/%m/%Y"):
        try:
            return datetime.strptime(value, fmt)
        except ValueError:
            continue
    return pd.NaT

In [17]:
# parse the date column
df['purchase_date'] = df['purchase_date'].astype(str).apply(parse_date)

In [18]:
df.head()

,customer_id,transaction_id,purchase_amount,currency,purchase_date,product_id,product_category,is_returned
0,25794,TXN2867825,120.50,USD,2023-03-15,P_234,books,no
2,86819,TXN5614226,-15.00,USD,2023-01-01,P_456,toys,True
3,64885,TXN5108603,-15.00,USD,2023-04-03,P_234,Books,True
4,16264,TXN4744854,45.99,USD,2023-04-03,P_456,Electronics,True
5,92385,TXN3341057,120.50,USD,NaT,P_234,TOYS,True


**Transform *product_id* Column**

In [20]:
# Check the number of unique values in the product_id

print(f"There are {df['product_id'].nunique()} unique values in the product_id column")


There are 4 unique values in the product_id column


In [21]:
# Convert product_id to category type
df['product_id'] = df['product_id'].astype('category')

**Transform *product_category* Column**

In [24]:
print(f"There are {df['product_category'].nunique()} values in the product category")
print(f'The values are {df['product_category'].unique()}')

There are 6 values in the product category
The values are ['books' 'toys' 'Books' 'Electronics' 'TOYS' 'electronics']


Although there are only three actual categories — Books, Toys, and Electronics — pandas' ase sensitivity has resulted in six distinct values. We will standardize these entries to ensure consistency.

In [25]:
# standardize product_category
df['product_category'] = df['product_category'].astype(str).str.strip().str.lower()

In [26]:
# convert product_category to category
df['product_category'] = df['product_category'].astype('category')

**Transform *is_returned* Column**

This column is ideally meant to be boolean, but currently contains inconsistent values such as "yes", "no", "true", "false", and NaNs. We will clean and standardize these values by mapping them appropriately to True and False, and then convert the column’s data type to boolean.

In [30]:
# Standardize is_returned column

print("Unique values in is_returned column:", df['is_returned'].unique())


Unique values in is_returned column: ['no' 'True' 'False' 'yes' nan]


In [31]:
# create a new column 'returned'
df['returned'] = df['is_returned'].astype(str).str.strip().str.lower()

In [32]:
# map 'no' to False and 'yes' to True

mapping = {
    'no':'No',
    'yes': 'Yes',
    'False': 'No',
    'True': 'Yes'}

df['returned'] = df['is_returned'].map(mapping)

In [33]:
# convert 'returned' to boolean
df['_returned'] = df['returned'].map({'Yes': True, 'No': False})

In [34]:
df.drop(columns=['is_returned', 'returned'], inplace=True)
df.rename(columns={'_returned': 'is_returned'}, inplace=True)

In [35]:
df['is_returned'] = df['is_returned'].astype(bool)

After a series of transformations, we cleaned and standardized the is_returned feature, then converted its data type to bool, which is more memory-efficient than the previous object dtype.

In [38]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
Index: 71209 entries, 0 to 99999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   customer_id       71209 non-null  category      
 1   transaction_id    71209 non-null  string        
 2   purchase_amount   71209 non-null  float64       
 3   currency          47322 non-null  category      
 4   purchase_date     57064 non-null  datetime64[ns]
 5   product_id        57127 non-null  category      
 6   product_category  71209 non-null  category      
 7   is_returned       71209 non-null  bool          
dtypes: bool(1), category(4), datetime64[ns](1), float64(1), string(1)
memory usage: 7.2 MB


The memory usage dropped significantly—from over 30 MB to approximately 7 MB—indicating that the transformations applied so far have been effective in optimizing memory. 

In [39]:
# reorder columns
df = df[['customer_id', 'transaction_id', 'product_id', 'purchase_amount', 'purchase_date', 
         'product_category', 'currency', 'is_returned']]

In [40]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
Index: 71209 entries, 0 to 99999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   customer_id       71209 non-null  category      
 1   transaction_id    71209 non-null  string        
 2   product_id        57127 non-null  category      
 3   purchase_amount   71209 non-null  float64       
 4   purchase_date     57064 non-null  datetime64[ns]
 5   product_category  71209 non-null  category      
 6   currency          47322 non-null  category      
 7   is_returned       71209 non-null  bool          
dtypes: bool(1), category(4), datetime64[ns](1), float64(1), string(1)
memory usage: 7.2 MB


In [41]:
df.head()

,customer_id,transaction_id,product_id,purchase_amount,purchase_date,product_category,currency,is_returned
0,25794,TXN2867825,P_234,120.50,2023-03-15,books,USD,False
2,86819,TXN5614226,P_456,-15.00,2023-01-01,toys,USD,True
3,64885,TXN5108603,P_234,-15.00,2023-04-03,books,USD,True
4,16264,TXN4744854,P_456,45.99,2023-04-03,electronics,USD,True
5,92385,TXN3341057,P_234,120.50,NaT,toys,USD,True


The primary objective of this challenge was to optimize memory efficiency during data loading. Through a series of transformations, we’ve successfully reduced memory usage. However, to further enhance performance—especially when working with large datasets—we will implement chunking: the process of reading data in smaller, manageable portions. Chunking is a proven technique for minimizing memory consumption. Next, I will create a script that reads the dataset in chunks and saves the processed output to a designated ***processed*** directory.